# 14 · GitHub release: build the public repository and push it
**Purpose:** collect the release from Drive into a clean folder, check it, and push it to GitHub. The token never leaves Colab Secrets.

**What goes in:**
- `notebooks/` 00–14 with outputs and Colab metadata stripped
- `config/config.json`
- `data/` (dataset, splits, attack sets, data card)
- `results/` (metrics, statistics, run registry, per-document predictions)
- `figures/`, `tables/`, `env/`

**Never included:**
- COLING-2025 texts (the `data/` upload and `external_*.parquet`)
- model checkpoints (`06_models`)
- raw API responses (`01_raw`)
- logs and the manuscript

**Before running (one time):**
1. On github.com create an **empty public repository** (no README, licence or .gitignore), e.g. `matd-turkic`.
2. Create a **fine-grained personal access token**: Settings → Developer settings → Fine-grained tokens.
   - Repository access: *Only select repositories* → that repository.
   - Permissions: *Contents: Read and write*.
   - Expiry: 7 days.
3. In Colab, click 🔑 (Secrets) and add `GITHUB_TOKEN` with the token. Enable notebook access.

**Run:** first with `DRY_RUN = True` (build + checks, nothing is pushed), read the report, then set `DRY_RUN = False` and run again.
Re-running later pushes a new commit that mirrors the current Drive state; files you added on GitHub by hand are removed.

**Runtime:** CPU, ~2–5 min.

In [ ]:
# ==== Settings (edit here only) ====
NOTEBOOK = "14_github_release"
PROJECT_ROOT = "/content/drive/MyDrive/multilingual-ai-text-detection"   # <- edit if needed
GH_OWNER = "hardkazakh"                 # your GitHub user name, e.g. "bsharimbayev"
GH_REPO = "matd-turkic"       # repository name (must already exist and be empty on first push)
BRANCH = "main"
DRY_RUN = True                # True: build + check only. False: also push.
INCLUDE_PREDICTIONS = True    # per-document scores (07_predictions): lets anyone rerun 09–13 without a GPU
AUTHOR = {"family-names": "Sharimbayev", "given-names": "Batyr", "orcid": "https://orcid.org/0009-0006-3323-231X",
          "affiliation": "SDU University, Kaskelen, Kazakhstan"}
MAX_FILE_MB = 95              # GitHub rejects files > 100 MB
MAX_TOTAL_MB = 900            # keep the repository well under 1 GB

import os, json, re, shutil, hashlib, base64, subprocess, datetime as dt
from pathlib import Path
try:
    from google.colab import drive; drive.mount("/content/drive")
except ImportError:
    pass
ROOT = Path(os.environ.get("MATD_ROOT", PROJECT_ROOT))
CFG = json.load(open(ROOT / "00_admin" / "config" / "config.json", encoding="utf-8"))
P = {k: ROOT / v for k, v in CFG["paths"].items()}
OUT = Path(os.environ.get("MATD_RELEASE_DIR", "/content/release")) / GH_REPO
if OUT.exists():
    shutil.rmtree(OUT)
OUT.mkdir(parents=True)
REPORT = []                                   # (level, message); any "FAIL" blocks the push
def note(level, msg):
    REPORT.append((level, msg)); print(f"[{level}] {msg}")
print(f"root={ROOT} | config v{CFG['config_version']} | data {CFG['data_version']} | out={OUT}")

## 1 · Notebooks: copy with outputs, execution counts and Colab metadata removed
Cell metadata is dropped entirely, because Colab stores the editor's display name and user ID there. Notebooks whose name starts with `_` (archived versions) are skipped.

In [ ]:
def clean_notebook(src, dst):
    nb = json.load(open(src, encoding="utf-8"))
    for c in nb.get("cells", []):
        c["metadata"] = {}
        if c.get("cell_type") == "code":
            c["outputs"] = []; c["execution_count"] = None
    nb["metadata"] = {"kernelspec": {"name": "python3", "display_name": "Python 3"}, "language_info": {"name": "python"},
                      "colab": {"provenance": []}}
    dst.parent.mkdir(parents=True, exist_ok=True)
    dst.write_text(json.dumps(nb, ensure_ascii=False, indent=1) + "\n", encoding="utf-8")

nbs = sorted(p for p in P["notebooks"].glob("*.ipynb") if not p.name.startswith("_"))
for p in nbs:
    clean_notebook(p, OUT / "notebooks" / p.name)
note("OK", f"{len(nbs)} notebooks cleaned: {', '.join(p.stem[:2] for p in nbs)}")
missing = [f"{i:02d}" for i in range(15) if not any(p.name.startswith(f"{i:02d}_") for p in nbs)]
if missing:
    note("WARN", f"notebooks missing on Drive: {missing}")

## 2 · Config, data, results, figures

In [ ]:
def copy(src, dst):
    dst.parent.mkdir(parents=True, exist_ok=True); shutil.copy2(src, dst)

def copy_tree(src_dir, dst_dir, patterns=("*",), exclude=lambda p: False):
    n = 0
    for pat in patterns:
        for f in sorted(Path(src_dir).rglob(pat)):
            if f.is_file() and not f.name.startswith(("_", ".")) and not exclude(f):
                copy(f, Path(dst_dir) / f.relative_to(src_dir)); n += 1
    return n

copy(ROOT / "00_admin" / "config" / "config.json", OUT / "config" / "config.json")

import pandas as pd
V = CFG["data_version"]
ds = pd.read_parquet(P["processed"] / f"dataset_{V}.parquet")
assert not ds.doc_id.str.startswith("colingkk").any(), "external texts found in the main dataset"
q_tr = set(ds[(ds.split == "train") & ds.qid.notna()].qid); q_te = set(ds[(ds.split == "test") & ds.qid.notna()].qid)
assert not (q_tr & q_te), "QID leakage between train and test"
(OUT / "data").mkdir(exist_ok=True)
ds.to_parquet(OUT / "data" / f"dataset_{V}.parquet", index=False)
import gzip                                   # mtime=0 → byte-identical file when the data are unchanged
with gzip.GzipFile(OUT / "data" / f"dataset_{V}.jsonl.gz", "wb", mtime=0) as gz:
    gz.write(ds.to_json(orient="records", lines=True, force_ascii=False).encode("utf-8"))
note("OK", f"dataset: {len(ds)} documents, {ds.lang.nunique()} languages, splits {ds.split.value_counts().to_dict()}")

att_p = P["attacks"] / f"attacks_{V}.parquet"
if att_p.exists():
    att = pd.read_parquet(att_p)
    att = att[~att.doc_id.str.startswith("colingkk")]
    att.to_parquet(OUT / "data" / f"attacks_{V}.parquet", index=False)
    note("OK", f"attack sets: {len(att)} rows, {att.eval_set.nunique()} evaluation sets")
else:
    note("WARN", "attack file not found")
for f in ("train_doc_ids.json", "val_doc_ids.json", "test_doc_ids.json", "split_summary.csv"):
    if (P["splits"] / f).exists():
        copy(P["splits"] / f, OUT / "data" / "splits" / f)
for f in ("generation_yield.csv", "meta_dropped.csv", "near_duplicates_removed.csv"):
    if (P["validated"] / f).exists():
        copy(P["validated"] / f, OUT / "data" / "build_logs" / f)
if (P["attacks"] / "attack_quality.csv").exists():
    copy(P["attacks"] / "attack_quality.csv", OUT / "data" / "build_logs" / "attack_quality.csv")

n = copy_tree(P["results"], OUT / "results" / "08_results", ("*.csv",))
n += copy_tree(P["stats"], OUT / "results" / "09_statistics", ("*.csv",))
if (P["registry"] / "run_registry.csv").exists():
    copy(P["registry"] / "run_registry.csv", OUT / "results" / "run_registry.csv"); n += 1
note("OK", f"results: {n} CSV files")
n = copy_tree(P["figures"], OUT / "figures", ("*.pdf", "*.png", "*.csv", "*.json"))
n += copy_tree(P["tables"], OUT / "tables", ("*.tex", "*.csv"))
note("OK", f"figures + tables: {n} files")
if INCLUDE_PREDICTIONS:
    n = copy_tree(P["predictions"], OUT / "results" / "predictions", ("*.parquet",))
    note("OK", f"predictions: {n} run files (doc_id + score only; COLING items are referenced by ID, without text)")
envs = sorted(P["env"].glob("env_*.json"))
latest = {}
for e in envs:                                 # keep the latest snapshot per notebook
    latest[re.sub(r"_\d{8}_\d{6}$", "", e.stem)] = e
for e in latest.values():
    copy(e, OUT / "env" / e.name)
note("OK", f"environment snapshots: {len(latest)}")

## 3 · README, data card, licences, citation
Numbers in the README are copied from the analysed results (Table 2 of the manuscript); update them if the results change.

In [ ]:
LANG_ROWS = "\n".join(f"| {k} | {v['name']} | {v['family']} | {v['script']} |" for k, v in CFG["languages"].items())
GEN_ROWS = "\n".join(f"| `{k}` | {v['model']} | {v['family']} |" for k, v in CFG["generators"].items())
split_counts = ds.groupby(["lang", "split"]).size().unstack().reindex(columns=["train", "val", "test"]).fillna(0).astype(int)
SPLIT_ROWS = "\n".join(f"| {l} | " + " | ".join(str(split_counts.loc[l, s]) for s in ["train", "val", "test"]) + " |" for l in split_counts.index)

README = f"""# Cross-lingual generalisation and robustness of multilingual AI-text detectors (Turkic languages)

Code, data and results for the paper **“Evaluating the Cross-Lingual Generalisation and Robustness of Multilingual AI Text Detectors: Evidence from Turkic Languages”** (under review).

We evaluate twelve detectors of machine-generated text on a new topic-paired corpus of **{len(ds):,} Wikipedia-style paragraphs**, written by humans or by five LLM families. The corpus covers four Turkic languages (Kazakh, Kyrgyz, Uzbek, Turkish) and two Indo-European reference languages (Russian, English). The language set crosses **language family × script**, so the two can be separated as drivers of cross-lingual transfer.

| Code | Language | Family | Script |
|---|---|---|---|
{LANG_ROWS}

## Main findings

The full analysis, with confidence intervals and multiplicity correction, is in the paper.

- **Fine-tuned multilingual encoders** (mDeBERTa-v3-base, XLM-R-base) reach test AUROC ≥ 0.987 in every language. Pooled AUROC is 0.995 for mDeBERTa and 0.992 for XLM-R.
- **Transfer pathway depends on the representation.** Transfer of monolingual detectors follows **language family** for the encoders and **script** for character n-gram models.
- **Unseen generator families** cost little (|ΔAUROC| ≤ 0.01), except Llama-3.3. Its drop of 0.04–0.05 is confounded with its atypical outputs.
- **Zero-shot detectors** (Binoculars, Fast-DetectGPT, log-likelihood, log-rank) are weaker, with pooled AUROC 0.76–0.88.
  - They fall below chance under 15 % homoglyph or typo perturbation.
  - Unicode NFKC normalisation with confusable mapping recovers most of the homoglyph loss.
- **A Kazakh-adapted scoring LM** does not help zero-shot detection. Much of its log-likelihood loss comes from per-token normalisation with its extended tokenizer.
- **Thresholds do not transfer across languages.** At a nominal 5 % FPR set on the training language, realised FPR in other languages reaches 89 %.

## Repository layout

```
notebooks/   00–14 Colab notebooks (the only code; outputs cleared)
config/      config.json – single source of truth for every setting
data/        dataset, splits, attack sets, build logs, data card (data/README.md)
results/     08_results/ metrics · 09_statistics/ RQ tests + posthoc/ · run_registry.csv · predictions/
figures/     all figures (PDF/PNG) and their source data
tables/      LaTeX tables
env/         package and GPU snapshots of the runs
```

## Reproducing the study

All code is in Colab notebooks that read and write a Google Drive folder, `MyDrive/multilingual-ai-text-detection/`. The folder layout is given in `config.json → paths`.

1. Copy `notebooks/` to `MyDrive/multilingual-ai-text-detection/notebooks/`.
2. Add Colab secrets:
   - `OPENROUTER_API_KEY` (needed for notebooks 03 and 05);
   - `HF_TOKEN` (optional).
3. Run the notebooks in order. Each one skips work that is already done, and every run is recorded in the run registry.

| # | Notebook | Runtime |
|---|---|---|
| 01 | setup, config, smoke tests | CPU |
| 02 | human texts from Wikipedia (revision as of {CFG['data']['wiki_cutoff_utc'][:10]}) | CPU |
| 03 | machine texts via OpenRouter | CPU (paid API) |
| 04 | pairing, normalisation, splits, shortcut audit, external set | CPU |
| 05 | attack sets (paraphrase, back-translation, homoglyphs, typos, defence) | GPU (T4+) + API |
| 06 | KerasHub fine-tuning (60 runs, batch 32, bf16) | A100 |
| 07 | character n-gram and stylometric baselines | CPU |
| 08 | zero-shot detectors (generic and Kazakh-adapted scorers) | A100 |
| 09–11 | evaluation, statistics, error analysis | CPU |
| 12 | figures and tables | CPU |
| 13 | exploratory robustness checks | CPU |
| 14 | this release | CPU |

**Re-running only the analysis (no GPU, no API):**

1. Put `config/config.json` in `00_admin/config/`, `data/dataset_{V}.parquet` in `03_processed/` and `data/attacks_{V}.parquet` in `05_attacks/`.
2. Put `data/splits/*` in `04_splits/`.
3. Put `results/predictions/*` in `07_predictions/` and `results/run_registry.csv` in `00_admin/registry/`.
4. Run notebooks 09–13.

The COLING-2025 Kazakh external test set is not redistributed. To reproduce those results, obtain it from the shared-task organisers and place it at `data/data/coling2025_kazakh.jsonl`.

**Frameworks:** Keras 3 and KerasHub (TensorFlow backend) for fine-tuning; Hugging Face Transformers only for the zero-shot scoring LMs; scikit-learn for the shallow models. Exact versions are in `env/`.

## Data

See [`data/README.md`](data/README.md) for fields, construction and licences. The data are released **for research on detecting machine-generated text**.

## Responsible use

Detector scores are probabilistic and can be wrong. Error rates depend on the language, the generator and on thresholds, which do not transfer between languages. Do not use any detector from this repository as the sole basis for decisions about individuals, for example in academic-misconduct cases.

## Licences

- **Code** (notebooks, config): Apache License 2.0 (`LICENSE`).
- **Data:** see `data/LICENSE.md`. Human texts are from Wikipedia and are released under CC BY-SA 4.0.

## Citation

See `CITATION.cff`. A full reference will be added when the paper is published.
"""

GEN_TABLE = "\n".join(f"| {g} | {n} |" for g, n in ds[ds.label == 1].generator.value_counts().sort_index().items())
DATA_CARD = f"""# Data card

## Contents

| File | What it is |
|---|---|
| `dataset_{V}.parquet` / `.jsonl.gz` | {len(ds):,} documents: {int((ds.label == 0).sum()):,} human + {int((ds.label == 1).sum()):,} machine, topic-paired |
| `attacks_{V}.parquet` | transformed **test** texts (paraphrase, back-translation, homoglyphs, typos, defence variants) |
| `splits/` | train / val / test document IDs and per-language counts |
| `build_logs/` | generation yield, removed items, attack quality (LaBSE similarity, language ID) |

### Languages and splits (documents)

| Language | train | val | test |
|---|---|---|---|
{SPLIT_ROWS}

### Generators (machine documents)

| Generator | Documents |
|---|---|
{GEN_TABLE}

Model identifiers, as requested through OpenRouter:

| ID | Model | Family |
|---|---|---|
{GEN_ROWS}

## Fields (`dataset_{V}`)

| Field | Description |
|---|---|
| `doc_id` | `{{pair_id}}:h` (human) or `{{pair_id}}:m` (machine) |
| `pair_id` | topic pair; each pair has one human and one machine paragraph on the same Wikipedia article |
| `lang` | ISO 639-1 code |
| `title`, `qid` | Wikipedia article title and Wikidata ID |
| `split` | `train` / `val` / `test`, assigned by hashing the QID. No test QID occurs in training in any language. |
| `label` | 0 = human, 1 = machine |
| `generator`, `family` | generator ID and family (`human` for human texts) |
| `pair_generator`, `pair_family` | generator of the pair's machine text (also set on the human row) |
| `prompt_id` | which of the two COLING-2025-style prompt templates was used |
| `text` | normalised paragraph (40–200 words) |
| `n_words` | word count |
| `source_meta` | JSON. Human: Wikipedia `revid` and timestamp. Machine: requested model, served model and provider. |

`attacks_{V}` adds `orig_doc_id` (the clean test document), `eval_set` (e.g. `T1_para`, `T2_bt`, `T3_h15`, `T3_h15both`, `T3_h15_def`, `T4_t05`, `test_def`), `attack`, `level`, `labse_sim` and `lid_ok`. Sets named `…both` contain transformed **human** texts as well.

## Construction

1. **Human texts.** The lead paragraph of each Wikipedia article, from the revision current on {CFG['data']['wiki_cutoff_utc'][:10]} (before ChatGPT). Paragraphs of 40–200 words whose language GlotLID confirms; bot stub templates are removed.
2. **Machine texts.** One paragraph per topic, generated from the article title alone with two prompt templates (temperature {CFG['generation']['temperature']}, top-p {CFG['generation']['top_p']}). Outputs that are too short, in the wrong language, or contain meta-commentary are rejected.
3. **Pairing and normalisation.**
   - Pairs whose lengths differ by more than 10 words or 25 % are dropped.
   - Both classes go through the same normaliser: NFC, zero-width and stress-mark removal, removal of headings and Markdown, quote and dash unification, and symmetric removal of parenthetical spans.
   - Near-duplicates are removed with MinHash.
   - Every (language, generator) cell is balanced to 195 pairs.
4. **Transformations** (test machine texts; homoglyphs and typos also on human texts):
   - paraphrase by Qwen3-235B-A22B;
   - back-translation with NLLB-200-distilled-600M;
   - Cyrillic↔Latin homoglyphs at 5 % / 15 %;
   - character typos at 5 % / 15 % of words.

## Known limitations

- **Single domain:** Wikipedia lead paragraphs only.
- **Stylometric signal:** punctuation and sentence-length features alone separate the classes with AUROC 0.78–0.90 (`results/08_results/data_audit.csv`).
- **Contamination:** the human texts are public and are likely part of the pre-training data of the generators and scorers.
- **API generators** may change behind the same model name.

## Licences and attribution

- **Human texts** and their transformed versions are adapted from Wikipedia. Wikipedia text is licensed under CC BY-SA (revisions from 2022 under CC BY-SA 3.0, which permits adaptations under CC BY-SA 4.0). They are released under **CC BY-SA 4.0**.
  - Attribution: each human text links to its source revision via `https://{{lang}}.wikipedia.org/w/index.php?oldid={{revid}}` (from `source_meta`). Authors are listed in each article's history.
- **Machine texts** are outputs of the listed commercial and open-weight models. Their use may be subject to the respective providers' terms. They are released together with the dataset under CC BY-SA 4.0 for research on detecting machine-generated text.
- **Back-translations** (`eval_set == "T2_bt"`) were produced with NLLB-200, whose model licence is CC BY-NC 4.0. Treat this subset as **non-commercial research use only**.
- **COLING-2025 Kazakh data** are **not included**. Predictions on them are referenced by ID only.

This card describes the data as released; it is not legal advice.
"""

DATA_LICENSE = """# Data licence

Unless stated otherwise in `data/README.md`, the files in `data/` are licensed under the
**Creative Commons Attribution-ShareAlike 4.0 International** licence (CC BY-SA 4.0):
https://creativecommons.org/licenses/by-sa/4.0/

Exception: the back-translated texts (`eval_set == "T2_bt"` in `attacks_*.parquet`) were produced with NLLB-200
(model licence CC BY-NC 4.0) and are provided for non-commercial research use only.

Human texts are adapted from Wikipedia; see `data/README.md` for attribution.
"""

CITATION = f"""cff-version: 1.2.0
message: "If you use this code or data, please cite it as below. A paper reference will be added on publication."
title: "Cross-lingual generalisation and robustness of multilingual AI-text detectors: code and data (Turkic languages)"
type: software
authors:
  - family-names: "{AUTHOR['family-names']}"
    given-names: "{AUTHOR['given-names']}"
    orcid: "{AUTHOR['orcid']}"
    affiliation: "{AUTHOR['affiliation']}"
license: Apache-2.0
repository-code: "https://github.com/{GH_OWNER}/{GH_REPO}"
date-released: "{dt.date.today().isoformat()}"
keywords: [machine-generated text detection, cross-lingual transfer, adversarial robustness, Turkic languages, Kazakh, low-resource languages]
"""

GITIGNORE = """.ipynb_checkpoints/
__pycache__/
*.pyc
.DS_Store
06_models/
*.keras
*.weights.h5
"""

(OUT / "README.md").write_text(README, encoding="utf-8")
(OUT / "data" / "README.md").write_text(DATA_CARD, encoding="utf-8")
(OUT / "data" / "LICENSE.md").write_text(DATA_LICENSE, encoding="utf-8")
(OUT / "CITATION.cff").write_text(CITATION, encoding="utf-8")
(OUT / ".gitignore").write_text(GITIGNORE, encoding="utf-8")
(OUT / ".gitattributes").write_text("*.parquet binary\n*.gz binary\n*.pdf binary\n*.png binary\n", encoding="utf-8")

import urllib.request
try:
    lic = urllib.request.urlopen("https://www.apache.org/licenses/LICENSE-2.0.txt", timeout=30).read().decode("utf-8")
    assert "Apache License" in lic and "Version 2.0" in lic
    (OUT / "LICENSE").write_text(lic, encoding="utf-8"); note("OK", "LICENSE (Apache-2.0) downloaded")
except Exception as e:
    note("FAIL", f"could not download the Apache-2.0 licence text ({e}); add LICENSE manually")
if not GH_OWNER:
    note("WARN", "GH_OWNER is empty: CITATION.cff has no repository URL, and pushing is impossible")

## 4 · Checks: secrets, personal data, excluded data, file sizes
Any **FAIL** blocks the push. Fix the cause on Drive (or in the settings) and run the notebook again.

In [ ]:
SECRET_PATTERNS = {
    "OpenRouter key": r"sk-or-v1-[0-9a-f]{20,}", "OpenAI key": r"sk-(?:proj-)?[A-Za-z0-9_-]{32,}",
    "Anthropic key": r"sk-ant-[A-Za-z0-9_-]{20,}", "Hugging Face token": r"hf_[A-Za-z0-9]{30,}",
    "GitHub token": r"(?:ghp|gho|ghu|ghs)_[A-Za-z0-9]{30,}|github_pat_[A-Za-z0-9_]{40,}",
    "Google API key": r"AIza[0-9A-Za-z_-]{35}", "private key": r"-----BEGIN [A-Z ]*PRIVATE KEY-----"}
EMAIL = re.compile(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}")
TEXT_EXT = {".ipynb", ".json", ".csv", ".md", ".tex", ".cff", ".txt", ".gitignore", ".gitattributes", ""}
tok_secret = None
try:
    from google.colab import userdata; tok_secret = userdata.get("GITHUB_TOKEN")
except Exception:
    tok_secret = os.environ.get("GITHUB_TOKEN")

hits, emails = [], set()
for f in OUT.rglob("*"):
    if not f.is_file():
        continue
    if f.suffix in TEXT_EXT:
        s = f.read_text(encoding="utf-8", errors="ignore")
    elif f.suffix == ".parquet":                           # scan text columns of tables too
        df_ = pd.read_parquet(f); s = "\n".join(df_[c].astype(str).str.cat(sep="\n") for c in df_.columns if df_[c].dtype == object)
    else:
        continue
    for name, pat in SECRET_PATTERNS.items():
        if re.search(pat, s):
            hits.append((name, str(f.relative_to(OUT))))
    if tok_secret and tok_secret in s:
        hits.append(("this GITHUB_TOKEN", str(f.relative_to(OUT))))
    if f.suffix in {".ipynb", ".json", ".csv", ".cff", ".md"}:
        emails |= {e for e in EMAIL.findall(s) if not e.endswith(("noreply.github.com", "example.com"))}
for name, path in hits:
    note("FAIL", f"possible {name} in {path}")
if not hits:
    note("OK", "no secrets found")
if emails:
    note("WARN", f"e-mail-like strings found (check they are meant to be public): {sorted(emails)[:10]}")

coling = [str(f.relative_to(OUT)) for f in (OUT / "data").rglob("*") if f.is_file() and ("coling" in f.name.lower() or f.name.startswith("external_"))]
if coling:
    note("FAIL", f"COLING/external text files must not be released: {coling}")
if att_p.exists() and (OUT / "data" / f"attacks_{V}.parquet").exists():
    a2 = pd.read_parquet(OUT / "data" / f"attacks_{V}.parquet")
    if a2.doc_id.str.startswith("colingkk").any():
        note("FAIL", "external texts found in the attack file")

sizes = {str(f.relative_to(OUT)): f.stat().st_size / 2**20 for f in OUT.rglob("*") if f.is_file()}
big = {k: round(v, 1) for k, v in sizes.items() if v > MAX_FILE_MB}
total = sum(sizes.values())
if big:
    note("FAIL", f"files over {MAX_FILE_MB} MB: {big}")
if total > MAX_TOTAL_MB:
    note("FAIL", f"release is {total:.0f} MB (> {MAX_TOTAL_MB} MB); set INCLUDE_PREDICTIONS = False or drop PNG figures")
note("OK", f"{len(sizes)} files, {total:.1f} MB in total; largest: " +
     ", ".join(f"{k} ({v:.1f} MB)" for k, v in sorted(sizes.items(), key=lambda x: -x[1])[:3]))

by_dir = {}
for k, v in sizes.items():
    top = k.split("/")[0] if "/" in k else "(root)"
    by_dir[top] = by_dir.get(top, 0) + v
print(pd.Series(by_dir).round(2).sort_values(ascending=False).to_string())
FAILS = [m for l, m in REPORT if l == "FAIL"]
print("\nRESULT:", "BLOCKED – fix the FAIL lines above" if FAILS else "ready to push")

## 5 · Push to GitHub
The token is read from Colab Secrets and passed only as an HTTP header for this one command. It is never written to the repository or the git config.
The commit author e-mail is your GitHub *no-reply* address, so no personal e-mail is published.

In [ ]:
def run(cmd, cwd=OUT, extra_env=None, secret=None):
    r = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True, env={**os.environ, **(extra_env or {})})
    out = (r.stdout + r.stderr)
    if secret:
        out = out.replace(secret, "***")
    return r.returncode, out.strip()

if DRY_RUN:
    print("DRY_RUN = True → nothing pushed. Read the report above, then set DRY_RUN = False and run the notebook again.")
elif FAILS:
    raise SystemExit("Push blocked by FAIL checks:\n- " + "\n- ".join(FAILS))
else:
    assert GH_OWNER, "set GH_OWNER"
    assert tok_secret, "add the Colab secret GITHUB_TOKEN and allow this notebook to access it"
    remote = os.environ.get("MATD_GIT_REMOTE", f"https://github.com/{GH_OWNER}/{GH_REPO}.git")
    if remote.startswith("https://github.com/"):
        req = urllib.request.Request(f"https://api.github.com/repos/{GH_OWNER}/{GH_REPO}",
                                     headers={"Authorization": f"Bearer {tok_secret}", "Accept": "application/vnd.github+json"})
        try:
            meta = json.load(urllib.request.urlopen(req, timeout=30))
            print(f"repository found: {meta['full_name']} | {'PRIVATE' if meta['private'] else 'PUBLIC'} | default branch {meta.get('default_branch')}")
        except Exception as e:
            raise SystemExit(f"Cannot access https://github.com/{GH_OWNER}/{GH_REPO} with this token ({e}). "
                             "Create the repository first and give the token access to it.")
    auth = base64.b64encode(f"x-access-token:{tok_secret}".encode()).decode()
    hdr = ["-c", f"http.extraHeader=Authorization: Basic {auth}"]
    for cmd in (["git", "init", "-q", "-b", BRANCH],
                ["git", "config", "user.name", f"{AUTHOR['given-names']} {AUTHOR['family-names']}"],
                ["git", "config", "user.email", f"{GH_OWNER}@users.noreply.github.com"],
                ["git", "remote", "add", "origin", remote]):
        code, out = run(cmd); assert code == 0, out
    code, out = run(["git", *hdr, "fetch", "-q", "origin", BRANCH], secret=auth)
    if code == 0:                                 # repository has history: commit on top of it (mirror the release tree)
        code, out = run(["git", "reset", "-q", "--soft", "FETCH_HEAD"]); assert code == 0, out
        print("existing history found → new commit on top of it")
    else:
        print("empty repository → first commit")
    run(["git", "add", "-A"])
    code, out = run(["git", "status", "--porcelain"])
    if not out:
        print("nothing changed since the last push")
    else:
        msg = f"Release: config v{CFG['config_version']}, data {V} ({dt.date.today().isoformat()})"
        code, out = run(["git", "commit", "-q", "-m", msg]); assert code == 0, out
        code, out = run(["git", *hdr, "push", "-u", "origin", BRANCH], secret=auth)
        print(out)
        if code != 0:
            raise SystemExit("push failed – see the message above (token permissions: Contents = Read and write).")
        print(f"\nDONE → https://github.com/{GH_OWNER}/{GH_REPO}")